# Lab 3: SQL Subqueries

## Setup Environment

In [ ]:
import pandas as pd
import duckdb

## Load Data

In [ ]:
books_df = pd.read_csv('books.csv')
sales_df = pd.read_csv('sales.csv')

## Create DuckDB Tables

In [ ]:
con = duckdb.connect(database=':memory:', read_only=False)
con.register('books', books_df)
con.register('sales', sales_df)

### 1. Simple Subquery

Find all books priced above the average price.

In [ ]:
query1 = '''
SELECT *
FROM books
WHERE price > (SELECT AVG(price) FROM books);
'''
con.execute(query1).fetchdf()

### 2. IN Subquery

Find titles of books authored by authors who have published more than 2 books.

In [ ]:
query2 = '''
SELECT title
FROM books
WHERE author IN (SELECT author FROM books GROUP BY author HAVING COUNT(*) > 2);
'''
con.execute(query2).fetchdf()

### 3. NOT IN Subquery

Find titles of books that have never been sold.

In [ ]:
query3 = '''
SELECT title
FROM books
WHERE book_id NOT IN (SELECT book_id FROM sales);
'''
con.execute(query3).fetchdf()

### 4. Subquery with Aggregation

List authors whose books have an average rating above the average rating of all books.

In [ ]:
query4 = '''
SELECT author
FROM books
GROUP BY author
HAVING AVG(rating) > (SELECT AVG(rating) FROM books);
'''
con.execute(query4).fetchdf()

### 5. Subquery with Comparison Operator

Retrieve the details of books priced higher than the average price of all books.

In [ ]:
query5 = '''
SELECT *
FROM books
WHERE price > (SELECT AVG(price) FROM books);
'''
con.execute(query5).fetchdf()

### 6. Nested Subquery

List the books with a price greater than the average price of books published by authors who have published exactly 1 book.

In [ ]:
query6 = '''
SELECT *
FROM books
WHERE price > (
    SELECT AVG(price)
    FROM books
    WHERE author IN (
        SELECT author
        FROM books
        GROUP BY author
        HAVING COUNT(*) = 1
    )
);
'''
con.execute(query6).fetchdf()

### 7. Subquery with Date Condition

Find titles of books that have been sold after the date when the most expensive book was first sold.

In [ ]:
query7 = '''
SELECT DISTINCT b.title
FROM books b
JOIN sales s ON b.book_id = s.book_id
WHERE s.sale_date > (
    SELECT s.sale_date
    FROM sales s
    JOIN books b ON s.book_id = b.book_id
    ORDER BY b.price DESC
    LIMIT 1
);
'''
con.execute(query7).fetchdf()

### 8. Correlated Subquery

Find each book's title along with the quantity sold in the most recent sale of that book.

In [ ]:
query8 = '''
SELECT b.title, s1.quantity
FROM books b
JOIN sales s1 ON b.book_id = s1.book_id
WHERE s1.sale_date = (
    SELECT MAX(s2.sale_date)
    FROM sales s2
    WHERE s2.book_id = s1.book_id
);
'''
con.execute(query8).fetchdf()

### 9. EXISTS Subquery

Find authors who have at least one book sold.

In [ ]:
query9 = '''
SELECT DISTINCT b.author
FROM books b
WHERE EXISTS (
    SELECT 1
    FROM sales s
    WHERE s.book_id = b.book_id
);
'''
con.execute(query9).fetchdf()

### 10. NOT EXISTS Subquery

Find authors who have no books sold.

In [ ]:
query10 = '''
SELECT DISTINCT author
FROM books b
WHERE NOT EXISTS (
    SELECT 1
    FROM sales s
    WHERE s.book_id = b.book_id
);
'''
con.execute(query10).fetchdf()